# Explore here

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD



import warnings

warnings.filterwarnings("ignore")

In [3]:
url =  "https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv"

In [4]:
df = pd.read_csv(url)

In [5]:
df.shape

(32561, 15)

In [6]:
df.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(24)

In [8]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [10]:
df['education'].value_counts()

education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Preschool          51
Name: count, dtype: int64

In [11]:
df['education.num'].value_counts()

education.num
9     10501
10     7291
13     5355
14     1723
11     1382
7      1175
12     1067
6       933
4       646
15      576
5       514
8       433
16      413
3       333
2       168
1        51
Name: count, dtype: int64

Procedemos a reemplazar las casillas con `?` por nulos para luego limpiar bien el dataset

In [12]:
df.replace("?", np.nan, inplace=True)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,NaN,77053,HS-grad,9,Widowed,NaN,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,NaN,186061,Some-college,10,Widowed,NaN,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,<=50K
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K


In [13]:
df.isnull().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     583
income               0
dtype: int64

In [14]:
df.dropna(inplace=True)

In [15]:
df.drop_duplicates(inplace=True)

# Filtrado Colaborativo basado en item

 seleccionamos las siguiente variables:
 `edad`
 `educacion`
 `occupation`
 `horas trabajadas` 
 `income`

In [16]:
variables = [
    "age",
    "education",
    "occupation",
    "hours.per.week",
    "income"
]

In [17]:
df = df[variables]

Procedemos a transformar las variables categoricas a numericas ya que los algoritmos van con numeros solamente

In [18]:
df_encoded = pd.get_dummies(
    df,
    columns=["education", "occupation"],
    drop_first=False
)

In [19]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df_encoded["age"] = scaler.fit_transform(
    df_encoded[["age"]])

# Definimos el problema de recomendacion

Queremos recomendar 
- niveles educativos
- ocupaciones 
- Cantidad de horas trabajadas
Para mejorar ingresos

In [21]:
perfil_usuario = {
    "age": 25,
    "education": "HS-grad",
    "occupation": "Sales",
    "hours.per.week": 20
}

In [22]:
high_income = df[df['income'] == '>50K']

In [23]:
high_income_encoded = pd.get_dummies(
high_income.drop("income", axis=1)
)

In [20]:
perfil_usuario = pd.DataFrame({
    "age": [25],
    "education": ["HS-grad"],
    "occupation": ["Sales"],
    "hours-per-week": [25]
})

In [25]:
rango_edad = 5
aspiracionales = high_income[
    (high_income['age'] >= perfil_usuario['age'] - rango_edad) & 
    (high_income['age'] <= perfil_usuario['age'] + rango_edad)
]

In [26]:
recom_educacion = aspiracionales['education'].mode()[0]
recom_ocupacion = aspiracionales['occupation'].mode()[0]
recom_horas = aspiracionales['hours.per.week'].median()

In [27]:
print(f"""
Para aumentar tus ingresos basado en personas de tu rango de edad:
- Mejora tu nivel educativo a: {recom_educacion}
- Considera cambiar a la ocupación: {recom_ocupacion}
- Deberías apuntar a trabajar unas {recom_horas} horas semanales.
""")


Para aumentar tus ingresos basado en personas de tu rango de edad:
- Mejora tu nivel educativo a: Bachelors
- Considera cambiar a la ocupación: Prof-specialty
- Deberías apuntar a trabajar unas 40.0 horas semanales.

